<a href="https://colab.research.google.com/github/Kaitokidbua/ASEAN_Transport/blob/main/ASEAN_Part1_Bangkok_Fig1-4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

✅ Setup complete — Dark theme loaded


In [14]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

Dataset: 2,027 rows | 5 cities
Cities: ['Bangkok', 'Jakarta', 'Kuala Lumpur', 'Singapore', 'Manila']


In [15]:
# ── Bangkok: prep data ────────────────────────────────────────────────────────
bkk = df[(df['City']=='Bangkok') & (df['Year'].between(2021,2025))].copy()
bkk['Mode_Label'] = bkk['Mode'].map(lambda x: MODE_BKK.get(x,x))

# monthly total
bkk_m = bkk.groupby('Date')['Ridership'].sum().reset_index()

# annual by mode — top 5
bkk_yr = bkk.groupby(['Year','Mode_Label'])['Ridership'].sum().reset_index()
top5_bkk = bkk.groupby('Mode_Label')['Ridership'].sum().nlargest(5).index.tolist()
bkk_top = bkk_yr[bkk_yr['Mode_Label'].isin(top5_bkk)]

# share total
bkk_share = bkk.groupby('Mode_Label')['Ridership'].sum().reset_index().sort_values('Ridership',ascending=False)

# YoY by mode
yoy_list = []
for mode in top5_bkk:
    sub = bkk_yr[bkk_yr['Mode_Label']==mode].sort_values('Year').copy()
    sub['YoY'] = sub['Ridership'].pct_change()*100
    yoy_list.append(sub)
bkk_yoy = pd.concat(yoy_list).dropna(subset=['YoY'])

print('Bangkok data ready ✅')

Bangkok data ready ✅


In [16]:
# ── Chart 1.1: Bangkok Monthly Trend ─────────────────────────────────────────
fig = px.area(
    bkk_m, x='Date', y='Ridership',
    title='<b>ตารางที่ 1.1</b>  กรุงเทพฯ — ผู้โดยสารรายเดือน ทุกระบบรวม (2021–2025)',
    labels={'Ridership':'จำนวนผู้โดยสาร','Date':''},
    color_discrete_sequence=[CITY_COLORS['Bangkok']],
)
fig.update_traces(fillcolor='rgba(255,107,107,0.13)', line_color=CITY_COLORS['Bangkok'], line_width=2.2)
fig.update_layout(**base_layout(420), hovermode='x unified',
                  yaxis=dict(tickformat='.2s',**ax_style()),
                  xaxis=ax_style())
fig.show()

In [17]:
import plotly.graph_objects as go

# ── Chart 1.2: Bangkok Mode Share Donut ──────────────────────────────────────
PIE_COLORS = ['#FF6B6B','#74B9FF','#4ECDC4','#FFB347','#C77DFF','#A8E6CF','#FD79A8']

fig = px.pie(
    bkk_share, names='Mode_Label', values='Ridership',
    title='<b>ตารางที่ 1.2</b>  กรุงเทพฯ — สัดส่วนผู้โดยสารแต่ละระบบ (2021–2025 รวม)',
    hole=0.45, color_discrete_sequence=PIE_COLORS,
)
fig.update_traces(textposition='inside', textinfo='percent+label',
                  insidetextorientation='radial', textfont_size=10,
                  marker=dict(line=dict(color=PAPER_BG, width=2)))

# Get the base layout configuration
layout_config = base_layout(460, legend_h=False)
# Update the legend properties as needed
layout_config['legend'].update(orientation='h', y=-0.15)

fig.update_layout(**layout_config)
fig.show()

In [18]:
# ── Chart 1.3: Bangkok Stacked Bar by Mode ───────────────────────────────────
fig = px.bar(
    bkk_top, x='Year', y='Ridership', color='Mode_Label',
    barmode='stack',
    title='<b>ตารางที่ 1.3</b>  กรุงเทพฯ — ผู้โดยสารแต่ละระบบขนส่ง รายปี (2021–2025)',
    labels={'Ridership':'ผู้โดยสาร','Mode_Label':'ระบบขนส่ง','Year':''},
    color_discrete_sequence=PIE_COLORS,
)
fig.update_layout(**base_layout(460),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()),
                  bargap=0.25)
fig.show()

In [19]:
# ── Chart 1.4: Bangkok YoY by Mode ───────────────────────────────────────────
fig = px.bar(
    bkk_yoy, x='Year', y='YoY', color='Mode_Label',
    barmode='group', text_auto='.0f',
    title='<b>ตารางที่ 1.4</b>  กรุงเทพฯ — อัตราการเปลี่ยนแปลงผู้โดยสาร YoY (%) แต่ละระบบ',
    labels={'YoY':'YoY Change (%)','Mode_Label':'ระบบขนส่ง','Year':''},
    color_discrete_sequence=PIE_COLORS,
)
fig.add_hline(y=0, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(460),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix='%',**ax_style()),
                  bargap=0.15)
fig.update_traces(textfont_size=8, textposition='outside')
fig.show()